## Filter Based Feature Selection
* VarianceThreshold
* SelectKBest
* SelectPercentile
* GenericUnivariateSelect

### 1. Variance Threshold 
* Remove Features with Variance below a certain threshold

In [ ]:
from sklearn.feature_selection import VarianceThreshold


### 2. Univariate Feature Selection
* (a) SelectKBest
    - Remove all but the k highest scoring features
* (b) SelectPercentile
    - Remove all but the user-specified highest scoring percentage of features
* (c) GenericUnivariateSelect
    - Performs univariate feature selection with a configurable strategy, which can be found via hyper-parameter search.

#### sklearn provides one more class of univariate feature selection methods that work on common univariate statistical tests for each feature:

* "SelectFpr" selects features based on a false positive rate test.

* "SelectFdr" selects features based on an estimated false discovery rate.

* "SelectFwe" selects features based on family-wise error rate.

#### Univariate scoring function

- Each API need a scoring function to score each feature.

- Three classes of scoring functions are proposed:
    * Mutual information (MI)  
    * Chi-square
    * F-statistics 
- MI and F-statistics can be used in both classification and regression problems.
    * For Mutual Information (MI)
        * mutual_info_regression
        * mutual_info_classif
    * For F-Statistics
        * f_regression
        * f_classif
- Chi-square can only be used in classification Problem
    * chi2

#### MI and chi-squared feature selection is recommended sparse data.





# 🧠 Univariate Feature Selection: The Brain (Scoring Functions)
**Core Concept:** Agar humare paas 10,000 columns (features) hain, toh machine learning model ko confuse hone se bachane ke liye hum har feature ka ek "Test" lete hain. Jo feature humare Target ($y$) ke sath sabse strong connection dikhata hai, hum usko highest "Marks" (Score) dete hain aur baakiyon ko nikal dete hain.

Scikit-learn (sklearn) hume 3 main tools (Scoring Functions) deta hai is connection ko test karne ke liye. Aaiye inko basic se advanced level tak samajhte hain.

---

## 1. F-Statistics (`f_classif`, `f_regression`)
**The "Straight-Line" Checker**

### 🟢 Basic Concept (Aasan Bhasha Me):
Ye tool sirf ek cheez dekhta hai: **"Kya Feature ($X$) aur Target ($y$) ke beech ek seedhi (linear) relationship hai?"** 
Jaise-jaise feature badhta hai, kya target bhi usi hisaab se badhta/ghatta hai? Agar connection seedhi line me hai, toh score high aayega. Agar connection complex ya curved hai, toh ye usko pakad nahi paayega aur score zero de dega.

### 🔴 High-Level Technical Detail:
Under the hood, F-statistics alag-alag statistical tests use karta hai:
*   **For Regression (`f_regression`):** Ye **Pearson Correlation Coefficient** calculate karta hai aur usko F-score me convert karta hai. Ye check karta hai ki variance kitna explain ho raha hai.
    *   Formula logic: $$F = \frac{R^2 / (1)}{ (1 - R^2) / (n - 2)}$$ (Jahan $R^2$ correlation hai aur $n$ samples hain).
*   **For Classification (`f_classif`):** Ye **ANOVA (Analysis of Variance)** test karta hai. Ye dekhta hai ki alag-alag classes ke beech mean (average) distance kitna zyada hai, aur ek hi class ke andar variance (bikhrav) kitna kam hai. 
    *   *Simple rule:* Agar "Dog" aur "Cat" class ka weight ek dusre se bohot alag hai, toh weight feature ka F-score high hoga.

**✅ Best Used For:**
Continuous Data (Numbers) jahan hume lagta hai relationship linear hai. Ye computationally **bohot fast** hota hai.

---

## 2. Mutual Information (`mutual_info_classif`, `mutual_info_regression`)
**The "Smart Detective" (Non-Linearity Master)**

### 🟢 Basic Concept (Aasan Bhasha Me):
Mutual Information (MI) ko seedhi line se koi farq nahi padta. Ye sirf ye puchta hai: **"Agar mujhe is Feature ($X$) ke baare me pata chal jaye, toh Target ($y$) ka guess marna kitna aasan ho jayega?"**
Chahe connection U-shape ka ho, Sine wave ho, ya zig-zag ho—agar unme koi bhi hidden pattern hai, MI usko dhoondh lega.

### 🔴 High-Level Technical Detail:
MI **Information Theory** aur **Shannon Entropy** par based hai. Entropy ka matlab hai data me kitni "uncertainty" (gadbadi ya randomness) hai.
*   **Formula:** $$MI(X, Y) = H(X) - H(X|Y)$$
    *   $H(X)$ = $X$ ki total gadbadi.
    *   $H(X|Y)$ = Agar $Y$ pata ho, toh $X$ me kitni gadbadi bachi.
*   **Sklearn Implementation:** Sklearn iske liye **k-Nearest Neighbors (k-NN)** approach use karta hai distances calculate karne ke liye (Kruskal-Wallis estimator for continuous data). 
*   **Scale:** Iska score hamesha $0$ ya usse bada hota hai. $0$ matlab dono me koi connection nahi (Independent). Higher number matlab strong connection.

**✅ Best Used For:**
Jab relationship complex ya Non-linear ho. Ye Text Data (NLP) aur Sparse data me bohot powerful hai. *(Note: Ye F-statistics se thoda slow run hota hai kyunki iski maths complex hai).*

---

## 3. Chi-Square (`chi2`)
**The "Category Matcher" (Expected vs Observed)**

### 🟢 Basic Concept (Aasan Bhasha Me):
Chi-Square sirf categories aur counts pe kaam karta hai. Ye test karta hai ki **"Jo data hume dikh raha hai (Observed) aur jo tukke se hona chahiye tha (Expected), unme kitna bada gap hai?"**
Maan lo hum Target "Gender" predict kar rahe hain. Feature hai "Likes Cosmetics". Agar "Cosmetics" aur "Female" hamesha ek sath dikhte hain (jo random chance se zyada hai), toh Chi-Square is feature ko high marks dega.

### 🔴 High-Level Technical Detail:
Ye ek statistical hypothesis test hai jo do Categorical variables ke beech **Independence** check karta hai.
*   **Mathematical Formula:** 
    $$\chi^2 = \sum \frac{(O_i - E_i)^2}{E_i}$$
    *   $O_i$ = Observed frequency (Actual counts)
    *   $E_i$ = Expected frequency (Agar dono me koi relation na hota toh count kya hota)
*   **Strict Condition:** Chi-square test me data hamesha **Non-Negative** hona chahiye (Yaani feature me koi negative number `-5`, `-2.3` nahi ho sakta). Ye usually frequencies (counts) ya probabilities par apply hota hai.

**✅ Best Used For:**
**Classification Tasks ONLY.** Specially jab aapke features Categorical hon, Boolean hon, ya Text processing me Bag-of-Words/TF-IDF ka output (jahan sirf positive numbers ya zeros hote hain).

---

## 🎯 Summary Rulebook for Industry:
1. **Target is Continuous Number (Regression)** + You want speed $\rightarrow$ Use `f_regression`
2. **Target is Continuous Number (Regression)** + Complex/Curved relationship $\rightarrow$ Use `mutual_info_regression`
3. **Target is Classes (Classification)** + Fast & Linear checks $\rightarrow$ Use `f_classif`
4. **Target is Classes (Classification)** + Complex relationship $\rightarrow$ Use `mutual_info_classif`
5. **Target is Classes (Classification)** + Features are Count/Categorical/Positive ONLY $\rightarrow$ Use `chi2`

# 🚀 The Ultimate Master Note: Univariate Feature Selection in Machine Learning

**Core Concept:** Jab humare paas hazaron features (columns) hote hain, toh sabko model mein daalna bewakoofi hai. Isse model slow hota hai, confuse hota hai, aur **Overfitting** ka shikar hota hai. 
**"Univariate"** ka matlab hai: Ek waqt mein sirf ek feature ko Target ($y$) ke sath check karna. Agar connection strong hai, toh rakho, warna kachre me daal do.

Is poore process ke 4 steps (ya pillars) hote hain. Aaiye ek-ek karke samajhte hain:

---

## Part 1: 🧠 The Brain (Scoring Functions)
Filter lagane se pehle, har feature ka "Test" lena padta hai taaki usko marks (score) mil sake. Sklearn iske liye 3 main tools deta hai:

### 1. F-Statistics (`f_classif`, `f_regression`)
*   **Aasan Bhasha:** Ye dekhta hai ki Feature aur Target ke beech koi **Seedhi Line (Linear)** ka connection hai ya nahi.
*   **Formula Logic:** Ye variance aur correlation pe kaam karta hai.
    $$F = \frac{R^2 / 1}{(1 - R^2) / (n - 2)}$$
*   **Kab Use Karein?** Jab data me linear connection ho aur aapko bohot fast calculation chahiye. 

### 2. Mutual Information (`mutual_info_classif`, `mutual_info_regression`)
*   **Aasan Bhasha:** Ye smart detective hai. Ye linear ya curved (U-shape, wave) har tarah ka connection pakad leta hai. Ye check karta hai ki Feature ko dekh kar Target ka guess marna kitna aasan hua.
*   **Kab Use Karein?** Complex real-world data, Text Data (NLP), ya Sparse data ke liye.

### 3. Chi-Square (`chi2`)
*   **Aasan Bhasha:** Ye dekhta hai ki jo counts hume dikh rahe hain, aur jo randomly aane chahiye the, unme kitna gap hai.
*   **Formula:** 
    $$\chi^2 = \sum \frac{(O_i - E_i)^2}{E_i}$$
*   **Kab Use Karein?** **Sirf Classification me**, aur tab jab features Categorical (jaise city, gender) ya strictly positive numbers hon.

---

## Part 2: 🛡️ The Bouncers (SelectKBest & SelectPercentile)
Ab marks mil gaye. Ab in features me se top rankers ko kaise chunein? Iske liye Filters use hote hain.

### 1. SelectKBest (The Fixed Bouncer)
**Logic:** "Bhai, saare features ko score do aur top 'K' (jaise top 5) features mujhe de do, baaki hata do."

```python
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectKBest, f_classif

# 1. Load Data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# 2. Filter lagana (Top 5 features chahiye)
selector = SelectKBest(score_func=f_classif, k=5)
X_selected_matrix = selector.fit_transform(X, y)

# 3. Best Practice: Columns ke naam wapas laana
selected_names = selector.get_feature_names_out(X.columns)
X_final = pd.DataFrame(X_selected_matrix, columns=selected_names)

print(f"Top 5 Selected Features: {list(selected_names)}")

### Mutual Information (MI)
- Measures dependency between two variables.
- It returns a non-negativevalue.

- MI = 0 for independent variables.
- Higher MI indicates higher dependency.

### Chi-square
- Measures dependence between two variables.
- Computes chi-square stats between non-negative feature (boolean or frequencies) and class label.
- Higher chi-square values indicates that the features and labels are likely to be correlated.

## SelectKBest (Sabse Zyada Use Hone Wala)
Concept: Ye bohot simple hai. Hum bas bolte hain: "Bhai, saare features ko score do aur top 'K' (jaise top 5 ya top 10) features mujhe de do, baaki sab hata do."

1. SelectKBest (The Exact Number Filter)

### 🟢 Basic Concept (Aasan Bhasha Me):
Ye filter bilkul ek strict competition ki tarah hai. Aap isko bolte ho: *"Bhai mujhe sirf Top 10 rankers chahiye."* Ye filter saare features ko unke marks ke hisaab se line me khada karta hai (highest to lowest), aur shuru ke 10 ko rakh kar baaki sabko hata deta hai.

### 🔴 High-Level Technical Detail:
*   **Mechanism:** Jab aap `.fit(X, y)` chalate hain, toh ye aapke diye hue scoring function (jaise `f_classif`) ko call karta hai. Score aane ke baad ye ek array banata hai aur `np.argsort()` (sorting algorithm) ka use karke top `k` indices nikal leta hai.
*   **The Mask (`get_support()`):** Under the hood, ye selected features ki ek boolean mask banata hai (jaise `[True, False, True]`). `.transform(X)` chalane par ye jahan `False` hota hai, us column ko drop kar deta hai.
*   **Tie-Breaker:** Agar 2 features ke score exact same aa jayein, toh sklearn unke original array index (jo pehle aaya wo pehle paaya) ke basis par tie break karta hai.

**✅ Best Used For:**
Jab aapko explicitly pata ho ki aapka model $K$ features ke sath best perform karta hai, ya jab computation power limited ho aur aapko strict dimension reduction karni ho (e.g., *mujhe sirf top 50 features hi RAM me load karne hain*).

In [4]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectKBest, f_classif

def select_k_best():
    # 1. dataset
    data = load_breast_cancer()
    X = pd.DataFrame(data.data, columns=data.feature_names)
    y = data.target
    
    print(f"Original Dataset Shape: {X.shape}") # (569 rows, 30 columns)
    
    # 2. Initialize SelectKBest
    # Score function 'f_classif' use kar rahe hain kyunki classification problem hai
    # k=5 ka matlab hai sirf Top 5 features chahiye
    selector = SelectKBest(score_func=f_classif, k=5)
    
    # 3. Fit and Transform
    X_selected_matrix = selector.fit_transform(X, y)
    
    # 4. BEST PRACTICE: Get selected feature names
    selected_feature_names = selector.get_feature_names_out(X.columns)
    
    # 5. Convert back to DataFrame
    X_final = pd.DataFrame(X_selected_matrix, columns=selected_feature_names)
    
    print(f"Selected Dataset Shape: {X_final.shape}")
    print(f"Top 5 Selected Features: {list(selected_feature_names)}")

# Run the function
select_k_best()

Original Dataset Shape: (569, 30)
Selected Dataset Shape: (569, 5)
Top 5 Selected Features: ['mean perimeter', 'mean concave points', 'worst radius', 'worst perimeter', 'worst concave points']


## 2. SelectPercentile (The Ratio Filter)

### 🟢 Basic Concept (Aasan Bhasha Me):
Kabhi-kabhi hume exact number nahi pata hota. Maan lo kal ko naya data aaya jisme 500 naye features hain, toh `K=10` fix rakhna bewakoofi hogi. Wahan hum `SelectPercentile` use karte hain. Aap isko bolte ho: *"Bhai, total jitne bhi features aayenge, mujhe unka Top 10% de dena."* Agar 100 features hain toh top 10 milenge, 1000 hain toh top 100 milenge.

### 🔴 High-Level Technical Detail:
*   **Mechanism:** Ye bhi `SelectKBest` ke engine par hi chalta hai, bas $K$ ki value dynamic hoti hai.
*   **Formula:** $K = \text{Total Features} \times \frac{\text{Percentile}}{100}$
*   **Robustness:** Production/Industry pipelines me ye `SelectKBest` se zyada robust maana jata hai kyunki ye dataset ki width badhne par auto-scale ho jata hai.

In [6]:
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.feature_selection import SelectPercentile, mutual_info_regression

def select_percentile():
    # 1. Create dataset (100 rows, 20 features, par kaam ke sirf 5 features hain, baaki kachra hai)
    X, y = make_regression(n_samples=100, n_features=20, n_informative=5, random_state=42)
    feature_names = [f"feature_{i}" for i in range(1, 21)]
    X_df = pd.DataFrame(X, columns=feature_names)
    
    print(f"Original Regression Dataset Shape: {X_df.shape}")
    
    # 2. Initialize SelectPercentile
    # Mutual Information use kar rahe hain (Regression ke liye)
    # percentile=25 ka matlab total 20 features ka 25% (yani top 5 features) select karo
    selector = SelectPercentile(score_func=mutual_info_regression, percentile=25)
    
    # 3. Fit and Transform
    X_selected_matrix = selector.fit_transform(X_df, y)
    
    # 4. Get names and format
    selected_feature_names = selector.get_feature_names_out(X_df.columns)
    X_final = pd.DataFrame(X_selected_matrix, columns=selected_feature_names)
    
    print(f"Selected Dataset Shape: {X_final.shape}")
    print(f"Top 25% Selected Features: {list(selected_feature_names)}")

select_percentile()

Original Regression Dataset Shape: (100, 20)
Selected Dataset Shape: (100, 5)
Top 25% Selected Features: ['feature_3', 'feature_4', 'feature_12', 'feature_13', 'feature_20']


## Part 3: The Strict Judges (FPR, FDR, FWE)
Ye Rank nahi dekhte, ye **P-Value (Sachai)** dekhte hain. P-value batati hai ki kya connection tukke (random chance) se toh nahi lag raha?

*   **SelectFpr (False Positive Rate):** Cutoff set karta hai (jaise `alpha=0.05`). Jiska P-value isse kam, wo pass.
*   **SelectFwe (Family-Wise Error Rate):** Sabse strict. Ek bhi kachra feature andar nahi aana chahiye (Medical data me use hota hai).
*   **SelectFdr (False Discovery Rate):** Thoda smart. Selected features me max 5% kachra allow karta hai (Genomics/Bio-informatics me use hota hai).


In [7]:
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectFpr, f_classif

# 1. Dataset me bohot sara noise dala
X, y = make_classification(n_samples=500, n_features=15, n_informative=3, random_state=1)
X_df = pd.DataFrame(X, columns=[f"Col_{i}" for i in range(1, 16)])

# 2. FPR lagaya (99% confidence chahiye, alpha=0.01)
selector = SelectFpr(score_func=f_classif, alpha=0.01)
X_strict = selector.fit_transform(X_df, y)

strict_names = selector.get_feature_names_out(X_df.columns)
print(f"Asli Features based on p-value: {list(strict_names)}")

Asli Features based on p-value: ['Col_6', 'Col_8', 'Col_13', 'Col_14']


### GenericUnivariateSelect
Maan lo aapko nahi pata ki KBest lagau, Percentile lagau ya FPR lagau. Toh kya karein?
Industry me hum GenericUnivariateSelect ko GridSearchCV ke sath laga dete hain taaki Machine khud decide kare ki sabse best strategy aur accuracy kisse mil rahi hai.

In [9]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.feature_selection import GenericUnivariateSelect, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

# 1. Load Data
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# 2. Pipeline banayi (Pehle selector, fir Model)
pipeline = Pipeline([
    ('feature_selector', GenericUnivariateSelect(score_func=f_classif)),
    ('model', RandomForestClassifier(random_state=42))
])

# 3. Model ko Menu de diya ki khud sab try karke batao!
param_grid = {
    'feature_selector__mode': ['k_best', 'percentile'],
    'feature_selector__param': [3, 5, 10, 20] 
}

# 4. Search Start
grid = GridSearchCV(pipeline, param_grid, cv=3)
grid.fit(X, y)

print(" --- Winner Strategy Results --- ")
print(f"Best Mode : {grid.best_params_['feature_selector__mode']}")
print(f"Best Value: {grid.best_params_['feature_selector__param']}")
print(f"Accuracy  : {grid.best_score_:.2f}")

c:\Users\sumit\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:782: UserWarning: k=20 is greater than n_features=13. All the features will be returned.
  warnings.warn(
c:\Users\sumit\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:782: UserWarning: k=20 is greater than n_features=13. All the features will be returned.
  warnings.warn(
c:\Users\sumit\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:782: UserWarning: k=20 is greater than n_features=13. All the features will be returned.
  warnings.warn(


 --- Winner Strategy Results --- 
Best Mode : k_best
Best Value: 20
Accuracy  : 0.97


c:\Users\sumit\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:782: UserWarning: k=20 is greater than n_features=13. All the features will be returned.
  warnings.warn(
